# 02 — Nettoyage des données

**Projet** : FakeNews Analyzer — DevComplex  
**Version** : V1 (local PySpark)  
**Objectif** : Nettoyer chaque source (HTML, URLs, stopwords, doublons) et sauvegarder en Parquet.

**Exécuter après** : `01_exploration.ipynb`

---

In [ ]:
# ============================================================
# Fichier  : 02_cleaning.ipynb
# Rôle     : Nettoyage par source (HTML, URLs, stopwords...)
# Version  : V1 (local)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================

import sys
import os
sys.path.insert(0, '..')

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from spark_utils import get_spark_session, load_raw_sources, show_label_distribution, save_parquet

spark = get_spark_session(app_name='FakeNews-Cleaning', memory='8g')
RAW_DIR = '../09_data/raw'
PROCESSED_DIR = '../09_data/processed'

sources = load_raw_sources(spark, RAW_DIR)
print(f'Sources disponibles : {list(sources.keys())}')

## Section 1 — Fonctions de nettoyage

In [ ]:
# Stopwords anglais (150 mots les plus fréquents — hardcodés pour éviter la dépendance NLTK en Spark)
ENGLISH_STOPWORDS = [
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','will','would','could','should','may','might','shall','can','need',
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'he','him','his','himself','she','her','hers','herself','it','its','itself',
    'they','them','their','theirs','themselves','what','which','who','whom',
    'this','that','these','those','am','if','else','each','both','few','more',
    'most','other','some','such','no','nor','not','only','own','same','so',
    'than','too','very','just','because','as','until','while','about','against',
    'between','into','through','during','before','after','above','below','from',
    'up','down','out','off','over','under','again','further','then','once'
]

import re

def clean_text_python(text) -> str:
    """
    Nettoyage identique à celui utilisé dans nlp_service.py (inférence API).
    La cohérence train/inférence est essentielle pour éviter le data leakage.
    """
    if not text or not isinstance(text, str):
        return None
    
    # Supprimer HTML, URLs, mentions @/#
    t = re.sub(r'<[^>]+>', ' ', text)
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'[@#]\w+', ' ', t)
    
    # Minuscules + supprimer non-alphabétique
    t = t.lower()
    t = re.sub(r'[^a-z\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    
    # Filtrer les tokens courts et les stopwords
    tokens = [tok for tok in t.split() if len(tok) >= 3 and tok not in ENGLISH_STOPWORDS]
    
    result = ' '.join(tokens)
    return result if result else None

# Enregistrer comme UDF Spark
clean_udf = F.udf(clean_text_python, StringType())

print('✓ UDF de nettoyage enregistrée')

## Section 2 — Application du nettoyage sur chaque source

In [ ]:
cleaned_sources = {}

for name, df in sources.items():
    print(f'\n── Nettoyage : {name} ──────────────────────────────')
    
    text_col = next((c for c in df.columns if c.lower() in ('text', 'statement', 'tweet', 'content')), None)
    label_col = next((c for c in df.columns if c.lower() in ('label', 'fraudulent')), None)
    
    if not text_col:
        print(f'  ⚠ Pas de colonne texte détectée — ignoré')
        continue
    
    before = df.count()
    print(f'  Lignes avant : {before:,}')
    
    df_clean = df.withColumn('clean_text', clean_udf(F.col(text_col)))
    
    keep_cols = ['clean_text']
    if label_col:
        keep_cols.append(label_col)
        if label_col != 'label':
            df_clean = df_clean.withColumnRenamed(label_col, 'label')
            keep_cols = ['clean_text', 'label']
    
    df_clean = df_clean.select(keep_cols)
    cleaned_sources[name] = df_clean
    print(f'  ✓ Nettoyage appliqué | colonnes : {df_clean.columns}')

## Section 3 — Suppression des lignes vides après nettoyage

In [ ]:
for name in list(cleaned_sources.keys()):
    df = cleaned_sources[name]
    before = df.count()
    
    df = df.filter(F.col('clean_text').isNotNull())
    df = df.filter(F.trim(F.col('clean_text')) != '')
    df = df.filter(F.size(F.split(F.col('clean_text'), ' ')) >= 5)
    
    after = df.count()
    print(f'[{name}] Vides supprimés : {before - after:,} ({(before-after)/before*100:.1f}%) → {after:,} lignes')
    cleaned_sources[name] = df

## Section 4 — Suppression des doublons exacts

In [ ]:
for name in list(cleaned_sources.keys()):
    df = cleaned_sources[name]
    before = df.count()
    df = df.dropDuplicates(['clean_text'])
    after = df.count()
    print(f'[{name}] Doublons supprimés : {before - after:,} → {after:,} lignes uniques')
    cleaned_sources[name] = df

## Section 5 — Stats avant/après nettoyage

In [ ]:
import pandas as pd

stats = []
for name, df in cleaned_sources.items():
    total = df.count()
    if 'label' in df.columns:
        label_dist = df.groupBy('label').count().toPandas()
        fake_count = label_dist[label_dist['label'] == 1]['count'].sum() if 1 in label_dist['label'].values else 0
        real_count = label_dist[label_dist['label'] == 0]['count'].sum() if 0 in label_dist['label'].values else 0
    else:
        fake_count = real_count = 'N/A'
    
    avg_len = df.select(F.avg(F.size(F.split(F.col('clean_text'), ' '))).alias('avg')).collect()[0]['avg']
    stats.append({
        'source': name, 'total': total,
        'FAKE': fake_count, 'REAL': real_count,
        'avg_words': round(avg_len, 1) if avg_len else 0
    })

print('\n=== STATS APRÈS NETTOYAGE ===')
print(pd.DataFrame(stats).to_string(index=False))

## Section 6 — Sauvegarde en Parquet

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

for name, df in cleaned_sources.items():
    output_path = os.path.join(PROCESSED_DIR, f'cleaned_{name}.parquet')
    df_with_source = df.withColumn('source', F.lit(name))
    save_parquet(df_with_source, output_path)

print('\n✓ Tous les fichiers nettoyés sauvegardés dans', PROCESSED_DIR)
print('Prochaine étape : exécuter 03_unification.ipynb')

In [ ]:
spark.stop()
print('✓ Session Spark arrêtée')